In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mikhail1681/walmart-sales/Walmart_Sales.csv


In [5]:

# Set the path to the file you'd like to load
file_path = "/kaggle/input/datasets/mikhail1681/walmart-sales/Walmart_Sales.csv"

df = pd.read_csv(file_path)

print("First 5 records:", df.head())

First 5 records:    Store        Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price  \
0      1  05-02-2010    1643690.90             0        42.31       2.572   
1      1  12-02-2010    1641957.44             1        38.51       2.548   
2      1  19-02-2010    1611968.17             0        39.93       2.514   
3      1  26-02-2010    1409727.59             0        46.63       2.561   
4      1  05-03-2010    1554806.68             0        46.50       2.625   

          CPI  Unemployment  
0  211.096358         8.106  
1  211.242170         8.106  
2  211.289143         8.106  
3  211.319643         8.106  
4  211.350143         8.106  


In [12]:

# Convert the Date column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Verify the change
print(df.dtypes)

Store                    int64
Date            datetime64[ns]
Weekly_Sales           float64
Holiday_Flag             int64
Temperature            float64
Fuel_Price             float64
CPI                    float64
Unemployment           float64
dtype: object


In [15]:
df = df.sort_values(by='Date')

In [16]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
5148,37,2010-02-05,536006.73,0,45.97,2.572,209.852966,8.554
2288,17,2010-02-05,789036.02,0,23.11,2.666,126.442065,6.548
4147,30,2010-02-05,465108.52,0,39.05,2.572,210.752605,8.324
3432,25,2010-02-05,677231.63,0,21.10,2.784,204.247194,8.187


In [17]:
df.tail()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
714,5,2012-10-26,319550.77,0,71.70,3.506,224.037814,5.422
6291,44,2012-10-26,361067.07,0,46.97,3.755,131.193097,5.217
5004,35,2012-10-26,865137.60,0,58.99,3.882,142.762411,8.665
5719,40,2012-10-26,921264.52,0,49.65,3.917,138.728161,4.145
6434,45,2012-10-26,760281.43,0,58.85,3.882,192.308899,8.667


In [19]:
df.size

51480

In [20]:
# Extract the year and count occurrences
year_counts = df['Date'].dt.year.value_counts().sort_index()

print(year_counts)

Date
2010    2160
2011    2340
2012    1935
Name: count, dtype: int64


# forcasting the sales for the year 2012 based on the sales of 2010 &2011

In [22]:
# Group by Date to get total sales across all stores per week
df_total = df.groupby('Date')['Weekly_Sales'].sum().reset_index()

# Split the data
train = df_total[df_total['Date'].dt.year < 2012]
test = df_total[df_total['Date'].dt.year == 2012]

*Random forest*

In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Create time-based features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week

# Define features (X) and target (y)
# We include Holiday_Flag, CPI, and Unemployment as they affect sales
features = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Month', 'Week']
X = df[features]
y = df['Weekly_Sales']

# Split: Train on 2010-2011, Test on 2012
train_mask = df['Year'] < 2012
test_mask = df['Year'] == 2012

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

In [24]:
# Initialize and fit the model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions for 2012
predictions = model.predict(X_test)

In [25]:
# Compare actual vs predicted
error = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error: ${error:,.2f}")

# Add predictions back to the test dataframe for plotting
df_2012 = df[test_mask].copy()
df_2012['Predicted_Sales'] = predictions

Mean Absolute Error: $156,308.97


In [30]:
# 6. Create the New Dataset
df_2012 = df[test_mask][['Date', 'Weekly_Sales']].copy()
df_2012.rename(columns={'Weekly_Sales': 'Actual_Sales'}, inplace=True)
df_2012['Predicted_Sales'] = predictions

# Save the final result
df_2012.to_csv('sales_forecast_2012.csv', index=False)

In [28]:
df_2012

,Date,Actual_Sales,Predicted_Sales
4533,2012-01-06,1099937.25,1.362334e+06
1816,2012-01-06,1865752.78,2.046635e+06
2102,2012-01-06,516087.65,5.369076e+05
5248,2012-01-06,558343.57,5.458143e+05
1244,2012-01-06,519585.67,5.993171e+05
...,...,...,...
714,2012-10-26,319550.77,3.268909e+05
6291,2012-10-26,361067.07,5.359514e+05
5004,2012-10-26,865137.60,8.747265e+05
5719,2012-10-26,921264.52,1.071241e+06
